# Crypto dataset for the Polymarket project

This notebook builds `crypto.csv`, one row per market, with the columns:

`slug, category, volume, close_date, price_7d, price_3d, price_1d, outcome`

Everything downloaded is kept in the `raw` folder, so a second run reuses it instead of downloading again.

## Decision log

Filters and outcomes:

* Downloaded the 1,000 highest-volume closed events tagged Crypto, and flattened them to one row per market — 9,821 candidate markets in total (unlike Politics, Crypto's biggest events are usually grouped "ladders" of many markets each, so restricting to one-market events would discard almost all of the highest-volume data)
* Of those 9,821 candidate markets:
  * Dropped markets also tagged with a higher-priority category (Crypto outranks every other category in this project, so this dropped 0)
  * Dropped 3,982 markets that closed outside 1 January to 12 November 2026
  * Dropped 15 markets with no closing time
  * Dropped 15 markets that did not resolve cleanly Yes or No (cancelled, settled 50/50, or not a Yes/No question)
  * 5,809 markets passed all of the above
* Took the outcome from the final prices
* Measured the 7, 3 and 1 day points back from the actual closing time (`closedTime`), not the scheduled end date
* Used hourly prices, and took the last price at or before each time point, as long as it was at most 24 hours old
* Dropped 55 of those 5,809 markets for missing one of the three prices
* Kept the 500 highest-volume markets that were left

In [1]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import pandas as pd
import requests

## Settings

The two Polymarket APIs:
- Gamma holds events, markets, tags and final outcomes
- CLOB holds the price histories.

In [2]:
GAMMA_ADDRESS = "https://gamma-api.polymarket.com"
CLOB_ADDRESS = "https://clob.polymarket.com"

The Crypto tag, and the tags that outrank it

In [3]:
CRYPTO_TAG_ID = 21
HIGHER_PRIORITY_TAG_IDS = []  # nothing outranks Crypto in this project

How much to download and what to measure:

- `EVENTS_PER_PAGE`: events per request (100 is the most the server allows).
- `PAGES_PER_ROUND`: how many more pages to fetch each time we run short.
- `MARKETS_WANTED`: rows in the final table.
- `DAYS_BEFORE_CLOSE`: the three time points before a market's close.

In [4]:
EVENTS_PER_PAGE = 100
PAGES_PER_ROUND = 10
MARKETS_WANTED = 500
DAYS_BEFORE_CLOSE = [7, 3, 1]

**Design choice.** A price only counts if it is at most this many hours old at the time point. This also drops markets that were open for less than 7 days.

In [5]:
MAX_PRICE_AGE_HOURS = 24

Date range for the closing date, written as `"YYYY-MM-DD"`

In [6]:
EARLIEST_CLOSE_DATE = "2026-01-01"
LATEST_CLOSE_DATE = "2026-11-12"

Where the raw downloads and the result are saved, the exact column order of the result, and the pause between two requests (to be polite to the server).

In [7]:
RAW_EVENTS_FOLDER = "raw/events"
RAW_HISTORIES_FOLDER = "raw/histories"
OUTPUT_FILE = "crypto.csv"
COLUMNS = ["slug", "category", "volume", "close_date",
           "price_7d", "price_3d", "price_1d", "outcome"]
PAUSE_SECONDS = 0.2

## Small helpers

`get_json` fetches an address and returns the reply as Python lists and dictionaries. If the server is busy (status 429 or 500 and above) it waits and tries again, up to 5 times. Other failures stop with an error.

In [8]:
def get_json(address, settings):
    for attempt in range(5):
        try:
            reply = requests.get(address, params=settings, timeout=30)
        except requests.RequestException:
            time.sleep(2 * (attempt + 1))
            continue
        if reply.status_code == 200:
            return reply.json()
        if reply.status_code == 429 or reply.status_code >= 500:
            time.sleep(2 * (attempt + 1))
            continue
        raise RuntimeError(f"{address} replied {reply.status_code}: {reply.text[:200]}")
    raise RuntimeError(f"{address} kept failing after 5 tries")

`read_file` and `write_file` load and save data as JSON files (both APIs return in JSON format)

`write_file` creates the folder first if it does not exist

In [9]:
def read_file(file_name):
    with open(file_name) as file:
        return json.load(file)


def write_file(data, file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    with open(file_name, "w") as file:
        json.dump(data, file)

`text_to_time` turns a time written as text into a real time in UTC in two formats: `2024-11-06T20:40:10Z` and `2024-11-06 15:17:41+00`.

In [10]:
def text_to_time(text):
    text = text.strip().replace(" ", "T")
    if text.endswith("Z"):
        text = text[:-1] + "+00:00"
    elif text.endswith("+00"):
        text = text + ":00"
    return datetime.fromisoformat(text).astimezone(timezone.utc)

The fields `outcomes`, `outcomePrices` and `clobTokenIds` arrive as text that looks like a list, e.g. `'["Yes", "No"]'`

`text_to_list` turns such text into a real list.

In [11]:
def text_to_list(text):
    if not text:
        return []
    return json.loads(text)

## Step 1: download the events

`fetch_events_page` asks for one page of closed crypto events, highest volume first

`cursor` is where the previous page ended (`None` for the first page)

It returns the page as the server sent it: `{"events": [...], "next_cursor": ...}`.

In [12]:
def fetch_events_page(cursor):
    settings = {
        "tag_id": CRYPTO_TAG_ID,
        "closed": "true",
        "order": "volume",
        "ascending": "false",
        "limit": EVENTS_PER_PAGE,
    }
    if cursor:
        settings["after_cursor"] = cursor
    return get_json(GAMMA_ADDRESS + "/events/keyset", settings)

`load_events` returns the events from the first `number_of_pages` pages. Each page is saved to its own file and only downloaded if the file is missing. When the server has no more pages it stops early.

Returns `(events, more_pages_exist)`.

In [13]:
def load_events(number_of_pages):
    events = []
    cursor = None
    for page_number in range(1, number_of_pages + 1):
        file_name = f"{RAW_EVENTS_FOLDER}/page_{page_number:04d}.json"
        if os.path.exists(file_name):
            page = read_file(file_name)
        else:
            page = fetch_events_page(cursor)
            write_file(page, file_name)
            time.sleep(PAUSE_SECONDS)
        events.extend(page.get("events", []))
        cursor = page.get("next_cursor")
        if not cursor:
            return events, False
    return events, True

## Step 2: build the market list

`market_outcome` works out how a market resolved, from its final prices. It finds where "Yes" sits in the outcomes list and uses that same position for the final price and the token ID.

Returns `(outcome, yes_token_id)`.

The outcome is 1 if Yes won, 0 if No won, and `None` if the market is unusable (not a Yes/No market, cancelled, or settled 50/50).

In [14]:
def market_outcome(market):
    outcomes = text_to_list(market.get("outcomes"))
    final_prices = text_to_list(market.get("outcomePrices"))
    token_ids = text_to_list(market.get("clobTokenIds"))

    if sorted(outcomes) != ["No", "Yes"]:
        return None, None
    if len(final_prices) != 2 or len(token_ids) != 2:
        return None, None

    yes_position = outcomes.index("Yes")
    yes_final_price = float(final_prices[yes_position])
    if yes_final_price == 1:
        return 1, token_ids[yes_position]
    if yes_final_price == 0:
        return 0, token_ids[yes_position]
    return None, None

`market_reason_to_drop` applies the filters from decision log to one market

Unlike Politics, Crypto's highest-volume events are usually grouped "ladders" with many markets each (e.g. one event with ten price-band markets) — restricting to events with exactly one market would throw away almost all of the highest-volume data. So here the filters apply **per market**, not per event: a market is kept or dropped on its own, and an event simply contributes all of its markets as separate candidates. The "higher-priority category" check still applies once per event, to every one of its markets, since tags belong to the event.

It returns `None` if the market passes, otherwise the reason it is dropped

In [15]:
def market_reason_to_drop(market):
    if not market.get("closedTime"):
        return "no closing time"

    close_date = text_to_time(market["closedTime"]).strftime("%Y-%m-%d")
    if EARLIEST_CLOSE_DATE is not None and close_date < EARLIEST_CLOSE_DATE:
        return "closed outside the date range"
    if LATEST_CLOSE_DATE is not None and close_date > LATEST_CLOSE_DATE:
        return "closed outside the date range"

    outcome, yes_token_id = market_outcome(market)
    if outcome is None:
        return "did not resolve cleanly Yes or No"
    return None

`build_market_list` turns the events into one record per usable market (slug, volume, closing time, Yes token ID, outcome), sorted from highest volume to lowest

It also counts how many markets were dropped for each reason

In [16]:
def build_market_list(events):
    markets = []
    drop_counts = {}
    slugs_seen = set()

    for event in events:
        tag_ids = [str(tag.get("id")) for tag in event.get("tags") or []]
        event_outranked = any(tag_id in tag_ids for tag_id in HIGHER_PRIORITY_TAG_IDS)

        for market in event.get("markets") or []:
            if event_outranked:
                reason = "also tagged with a higher-priority category"
            else:
                reason = market_reason_to_drop(market)
            if reason is None and market["slug"] in slugs_seen:
                reason = "duplicate"
            if reason is not None:
                drop_counts[reason] = drop_counts.get(reason, 0) + 1
                continue

            outcome, yes_token_id = market_outcome(market)
            slugs_seen.add(market["slug"])
            markets.append({
                "slug": market["slug"],
                "volume": float(market.get("volume") or 0),
                "closed_time": text_to_time(market["closedTime"]),
                "yes_token_id": yes_token_id,
                "outcome": outcome,
            })

    markets.sort(key=lambda market: market["volume"], reverse=True)
    return markets, drop_counts

## Step 3: download the price histories

`fetch_price_history` gets the hourly Yes prices (`fidelity` 60 = one point per 60 minutes) for the stretch we need: from just before the 7-day point up to the close

Each point is `{"t": seconds since 1970, "p": price}`

In [17]:
def fetch_price_history(market):
    longest_look_back = timedelta(days=max(DAYS_BEFORE_CLOSE), hours=MAX_PRICE_AGE_HOURS + 1)
    settings = {
        "market": market["yes_token_id"],
        "startTs": int((market["closed_time"] - longest_look_back).timestamp()),
        "endTs": int(market["closed_time"].timestamp()),
        "fidelity": 60,
    }
    reply = get_json(CLOB_ADDRESS + "/prices-history", settings)
    return reply.get("history", [])

`load_history` returns the market's price history, downloaded only if its file is missing.

In [18]:
def load_history(market):
    file_name = f"{RAW_HISTORIES_FOLDER}/{market['slug']}.json"
    if os.path.exists(file_name):
        return read_file(file_name)
    history = fetch_price_history(market)
    write_file(history, file_name)
    time.sleep(PAUSE_SECONDS)
    return history

## Step 4: build the table

`price_before_close` finds the last price at or before (closing time − `days_before`)

It returns `None` if there is no price there, or if the last one is older than `MAX_PRICE_AGE_HOURS`

In [19]:
def price_before_close(history, closed_time, days_before):
    target_seconds = (closed_time - timedelta(days=days_before)).timestamp()
    oldest_allowed = target_seconds - MAX_PRICE_AGE_HOURS * 3600

    best_point = None
    for point in history:
        if oldest_allowed <= point["t"] <= target_seconds:
            if best_point is None or point["t"] > best_point["t"]:
                best_point = point
    if best_point is None:
        return None
    return best_point["p"]

`build_rows` goes down the markets from highest volume and makes a row for each one that has all three prices

It stops once `MARKETS_WANTED` rows are made, and prints progress every 50 markets

Returns `(rows, missing_price_count)`

In [20]:
def build_rows(markets):
    rows = []
    missing_price_count = 0

    for position, market in enumerate(markets, start=1):
        if len(rows) == MARKETS_WANTED:
            break
        history = load_history(market)
        prices = [price_before_close(history, market["closed_time"], days)
                  for days in DAYS_BEFORE_CLOSE]
        if None in prices:
            missing_price_count += 1
            continue
        rows.append({
            "slug": market["slug"],
            "category": "Crypto",
            "volume": market["volume"],
            "close_date": market["closed_time"].strftime("%Y-%m-%d"),
            "price_7d": prices[0],
            "price_3d": prices[1],
            "price_1d": prices[2],
            "outcome": market["outcome"],
        })
        if position % 50 == 0:
            print(f"  checked {position} markets, {len(rows)} rows so far")

    return rows, missing_price_count

`check_table` stops with a error if the table breaks the plan's format:
- wrong columns or order
- blank cells
- a repeated slug
- a price outside 0 to 1
- an outcome that is not 0 or 1
- a category other than Crypto

In [21]:
def check_table(table):
    assert list(table.columns) == COLUMNS, "wrong columns or wrong order"
    assert not table.isnull().any().any(), "there are blank cells"
    assert not table["slug"].duplicated().any(), "a slug appears twice"
    for column in ["price_7d", "price_3d", "price_1d"]:
        assert table[column].between(0, 1).all(), f"{column} has a value outside 0 to 1"
    assert table["outcome"].isin([0, 1]).all(), "outcome must be 0 or 1"
    assert (table["category"] == "Crypto").all(), "category must be Crypto"

## Run everything

`main` runs the four stages in order. If fewer than `MARKETS_WANTED` rows come out and the server still has more events, it fetches `PAGES_PER_ROUND` more pages and tries again. Then it checks the table, saves it and prints a summary

In [22]:
def main():
    number_of_pages = PAGES_PER_ROUND

    while True:
        events, more_pages_exist = load_events(number_of_pages)
        markets, drop_counts = build_market_list(events)
        print(f"{number_of_pages} pages: {len(events)} events, "
              f"{len(markets)} markets pass the filters")

        rows, missing_price_count = build_rows(markets)
        if len(rows) >= MARKETS_WANTED or not more_pages_exist:
            break
        number_of_pages += PAGES_PER_ROUND

    table = pd.DataFrame(rows, columns=COLUMNS)
    check_table(table)
    table.to_csv(OUTPUT_FILE, index=False)

    print()
    print("Events downloaded:        ", len(events))
    for reason, count in sorted(drop_counts.items()):
        print(f"  dropped, {reason}: {count}")
    print("Markets passing filters:  ", len(markets))
    print("Dropped, missing a price: ", missing_price_count)
    print("Rows saved:               ", len(table), "to", OUTPUT_FILE)
    if len(table) < MARKETS_WANTED:
        print("WARNING: ran out of events before reaching", MARKETS_WANTED, "rows")

Run main

In [23]:
main()

10 pages: 1000 events, 5809 markets pass the filters


  checked 50 markets, 49 rows so far


  checked 100 markets, 97 rows so far


  checked 150 markets, 144 rows so far


  checked 200 markets, 184 rows so far


  checked 250 markets, 229 rows so far


... (3 outputs left out)

  checked 450 markets, 404 rows so far


  checked 500 markets, 449 rows so far


  checked 550 markets, 496 rows so far



Events downloaded:         1000
  dropped, closed outside the date range: 3982
  dropped, did not resolve cleanly Yes or No: 15
  dropped, no closing time: 15
Markets passing filters:   5809
Dropped, missing a price:  55
Rows saved:                500 to crypto.csv


## Result

In [24]:
pd.read_csv(OUTPUT_FILE)

,slug,category,volume,close_date,price_7d,price_3d,price_1d,outcome
0,microstrategy-sells-any-bitcoin-by-may-31-2026,Crypto,3.758131e+08,2026-06-04,0.0705,0.1230,0.0085,0
1,lighter-market-cap-fdv-1b-one-day-after-launch,Crypto,5.505670e+07,2026-01-03,0.9250,0.9975,0.9995,1
2,will-satoshi-move-any-bitcoin-in-2025,Crypto,4.418079e+07,2026-01-01,0.0035,0.0045,0.0025,0
3,will-bitcoin-reach-1000000-by-december-31-2025,Crypto,3.139461e+07,2026-01-01,0.0005,0.0005,0.0005,0
4,will-bitcoin-reach-150k-in-february-2026,Crypto,2.882922e+07,2026-03-01,0.0015,0.0005,0.0005,0
...,...,...,...,...,...,...,...,...
495,bitcoin-above-70k-on-april-1,Crypto,7.621884e+05,2026-04-01,0.5650,0.1350,0.0745,0
496,bitcoin-above-74k-on-april-10,Crypto,7.608446e+05,2026-04-10,0.0250,0.0375,0.1245,0
497,bitcoin-above-82k-on-february-3,Crypto,7.607501e+05,2026-02-03,0.9400,0.1910,0.0715,0
498,bitcoin-above-60k-on-march-31,Crypto,7.562343e+05,2026-03-31,0.9700,0.9795,0.9975,1
